# Pan-cancer model training

In [1]:
import time
import pandas as pd
from monte import Monte, train_with_cv

## Load data

**Merge training and validation set**

In [ ]:
df_meta_train = pd.read_csv("../../data/cancer-methyl/final_split/train_pan-cancer_meta.csv")
df_beta_train = pd.read_parquet("../../data/cancer-methyl/final_split/train_pan-cancer_beta.parquet")
df_meta_val = pd.read_csv("../../data/cancer-methyl/final_split/val_pan-cancer_meta.csv")
df_beta_val = pd.read_parquet("../../data/cancer-methyl/final_split/val_pan-cancer_beta.parquet")

In [3]:
df_beta = pd.concat([df_beta_train, df_beta_val])

df_meta = pd.concat([df_meta_train, df_meta_val])
df_meta = df_meta.set_index("Barcode", drop=False)
df_meta = df_meta.loc[df_beta.index]

we used the ESTIMATE as the primary metric for learning the cancer purity

In [4]:
df_meta_estimate = df_meta.dropna(subset=["ESTIMATE"])
df_beta_estimate = df_beta.loc[df_meta_estimate["Barcode"]]

In [5]:
len(df_meta), len(df_meta_estimate)

(4928, 4619)

**Test set**

In [ ]:
df_beta_test = pd.read_parquet("../../data/cancer-methyl/final_split/test_pan-cancer_beta.parquet")
df_meta_test = pd.read_csv("../../data/cancer-methyl/final_split/test_pan-cancer_meta.csv")
df_meta_test = df_meta_test.set_index("Barcode", drop=False)
df_meta_test = df_meta_test.loc[df_beta_test.index]

## Model training and prediction

Training pan-cancer monte model

In [7]:
start = time.time()
monte = train_with_cv(df_beta_estimate, df_meta_estimate["ESTIMATE"])
end = time.time()
training_time = end - start

Prediction on the test set. Note here we use `df_beta`, because we could impute the `NaN` in the ESTIMATE and evaluate with out metrics.

In [8]:
purity_pred = monte.predict_purity(df_beta)
purity_pred_test = monte.predict_purity(df_beta_test)

In [9]:
df_meta["pred_ESTIMATE"] = purity_pred
df_meta_test["pred_ESTIMATE"] = purity_pred_test

For timing purpose only

In [10]:
cancer_types = df_meta_test["Cancer.type"].unique()
df_cancer_pred_time = pd.DataFrame({
    "cancer": cancer_types,
    "prediction_time": [0.0] * len(cancer_types),
    "n_samples": [0] * len(cancer_types)
})

for i, cancer in enumerate(cancer_types):
    df_cancer = df_meta_test[df_meta_test["Cancer.type"] == cancer]
    df_cancer_beta = df_beta_test.loc[df_cancer["Barcode"]]
    
    start = time.time()
    monte.predict_purity(df_cancer_beta)
    end = time.time()
    
    prediction_time = end - start
    df_cancer_pred_time.loc[i, "prediction_time"] = prediction_time
    df_cancer_pred_time.loc[i, "n_samples"] = len(df_cancer)

## Save model and results

In [11]:
monte.save("../../data/monte_outputs/trained_models/pancancer_monte_model.pkl")

In [12]:
df_meta.to_csv("../../data/monte_outputs/pancancer/pancancer_meta_with_predictions.csv", index=False)
df_meta_test.to_csv("../../data/monte_outputs/pancancer/pancancer_meta_test_with_predictions.csv", index=False)

In [13]:
with open("../../data/monte_outputs/pancancer/pancancer_training_time.csv", "w") as f:
    f.write("Method,Training Time (seconds),n_samples\n")
    f.write(f"MONTE,{training_time},{len(df_beta_estimate)}\n")

In [14]:
df_cancer_pred_time.to_csv("../../data/monte_outputs/pancancer/pancancer_prediction_time_by_cancer.csv", index=False)